In [1]:
from ovo import db, storage, schedulers, descriptors, design_logic, descriptor_logic, \
    models_proteinqc, descriptors_proteinqc, project_logic
from ovo.app.components import molstar_notebook, StructureVisualization
import os
import seaborn as sns
from matplotlib import pyplot as plt

%config InlineBackend.figure_format = 'retina'

Registering plugin ovo_promb
Registering plugin ovo_proteindj


OVO home /home/username/ovo

In [2]:
project, project_round = project_logic.get_or_create_project_round("OVO Publication Examples 1", "Binder design")

In [3]:
pools = db.Pool.select(round_id=project_round.id)
for pool in pools:
    print(pool.id, pool.name)

rsj 4ZXB BindCraft default
avz 4ZXB 1000 designs default weights
mmo 4ZXB 1000 designs beta sheet
qki Top designs diversification


In [4]:
# Pool id from previous notebooks
POOL_IDS = ['avz', 'mmo', 'qki']

In [5]:
designs = db.Design.select(pool_id__in=POOL_IDS, accepted=True)
len(designs)

43

In [6]:
workflow = models_proteinqc.ProteinQCWorkflow(
    chains=['A'],
    design_ids=[design.id for design in designs],
    tools=['seq_composition', 'esm_1v', 'esm_if', 'dssp', 'peppatch', 'proteinsol'],
)
workflow.validate()
workflow.get_table_row()

Workflow  type    ProteinQC workflow
dtype: object

In [7]:
print(schedulers.keys())

SCHEDULER_KEY = 'slurm_singularity'

dict_keys(['slurm_singularity', 'pbs_singularity', 'local_singularity', 'local_conda', 'local_single_gpu'])


In [8]:
#
# SUBMIT - note that running this multiple times will submit a the workflow each time!
#
descriptor_job = descriptor_logic.submit_descriptor_workflow(
    workflow=workflow,
    scheduler_key=SCHEDULER_KEY,
    project_id=project.id
)
descriptor_job.id

Submitting workflow: nextflow run -with-trace trace.txt -work-dir /home/username/ovo/workdir/work /home/username/projects/ovo-open-source/ovo/ovo/pipelines/proteinqc --publish_dir output --reference_files_dir /home/username/ovo/reference_files --shared_modules ovo:/home/username/projects/ovo-open-source/ovo/ovo,ovo_promb:/home/username/projects/ovo-open-source/ovo-promb/ovo_promb,ovo_proteindj:/home/username/projects/ovo-proteindj/ovo_proteindj -config /home/username/projects/ovo-open-source/ovo/ovo/pipelines/nextflow_default.config -config /home/username/projects/ovo-open-source/ovo/ovo/pipelines/proteinqc/nextflow.config -profile singularity -config /home/username/ovo/nextflow_slurm_singularity.config -ansi-log false -bg --input_pdb /home/username/ovo/workdir/inputs/5a/385ca0da021fb0001d1f6be796053568350ee6/input_pdb_paths.txt --tools seq_composition,esm_1v,esm_if,dssp,peppatch,proteinsol --chains A --batch_size 50
Execution directory: /home/username/ovo/workdir/execdir/0cf46aea-c169

'9f5093'

In [9]:
descriptor_logic.process_results(descriptor_job)

Waiting for job 0cf46aea-c169-11f0-adc8-0248d8152725 to finish...


In [10]:
db.DescriptorValue.count(design_id__in=db.Design.select_values('id', pool_id__in=POOL_IDS, accepted=True))

6768

In [11]:
values = descriptor_logic.get_wide_descriptor_table(
    design_ids=db.Design.select_values('id', pool_id__in=POOL_IDS, accepted=True),
)
print(len(values), 'designs')
print(len(values.columns), 'descriptors')
values.head()

43 designs
117 descriptors


,Sequence A,AF2 iPAE,AF2 ipTM score,AF2 pTM score,AF2 Binder PAE,AF2 Binder pLDDT,AF2 Target-aligned Binder RMSD,AlphaFold2 Initial Guess prediction,Radius of gyration (backbone),N binder-target backbone contacts,...,Normalized positive top1 patch area at pH 7.4,Negative patch area at pH 7.4,Normalized negative patch area at pH 7.4,Negative top1 patch area at pH 7.4,Normalized negative top1 patch area at pH 7.4,Electrostatic volume integral at pH 7.4,Positive electrostatic regions volume integral at pH 7.4,Negative electrostatic regions volume integral at pH 7.4,Positive electrostatic volume integral at pH 7.4,Negative electrostatic volume integral at pH 7.4
design_id,,,,,,,,,,,,,,,,,,,,,
ovo_avz_0030_cycle02,EEEEKYEKLAEVALYGNELVESIKDEEEKEKLAKYVLEVIENREKI...,5.209966,0.827816,0.900388,4.536248,89.220834,0.821800,project/b8e657bb-b0b0-423e-9235-383a6d8f74e5/p...,11.558935,54,...,0.008814,2397.582566,0.517766,2397.582566,0.517766,-53204.939693,247.264672,-37802.189478,524.728685,-53729.668377
ovo_avz_0124_cycle04,KEEELEKLAYEYLEYSIKSYEYKKKAEELEKSEYEDEEKKKKEIEE...,5.833490,0.800033,0.887135,5.114184,91.134697,0.827681,project/b8e657bb-b0b0-423e-9235-383a6d8f74e5/p...,16.185386,47,...,0.002133,1329.886461,0.221943,494.351643,0.082502,-29282.013327,1603.259674,-20382.537688,5073.431641,-34355.444969
ovo_avz_0299_cycle01,MSIEEIKKILEEIKKEAEAKNAKKVAEAAAKDPELAKKLAAELTAE...,5.941155,0.804241,0.887639,5.177787,88.795263,0.914129,project/b8e657bb-b0b0-423e-9235-383a6d8f74e5/p...,15.665892,65,...,0.013346,1431.141721,0.264025,1222.315483,0.225499,-31439.230457,1221.440562,-21455.968271,3326.778095,-34766.008552
ovo_avz_0481_cycle03,MEAALKAAEEEKEFQKQVELAKLMIESYKKEGDEEQAEYWEKILEE...,6.987492,0.746239,0.876344,6.214309,85.058039,0.938373,project/b8e657bb-b0b0-423e-9235-383a6d8f74e5/p...,13.887859,35,...,0.004164,2322.248872,0.491070,2295.299667,0.485372,-62153.827602,169.365019,-47174.741642,462.410420,-62616.238021
ovo_avz_0732_cycle04,EEEEKEKKLKELEKQAKEVEEEGRKKVKEAEAKLKAGDKSEEVKEL...,9.498611,0.558957,0.809674,7.643191,83.656645,0.943637,project/b8e657bb-b0b0-423e-9235-383a6d8f74e5/p...,15.913069,49,...,0.026966,1342.739388,0.243623,1156.812464,0.209889,-18930.954449,6064.660410,-19659.415027,11991.801964,-30922.756413


In [12]:
values.to_csv('../../data/results/04_rfdiffusion_binder_design/descriptors.csv')

### Save descriptors for all designs including not accepted

In [13]:
all_values = descriptor_logic.get_wide_descriptor_table(
    design_ids=db.Design.select_values('id', pool_id__in=POOL_IDS),
    descriptor_keys=[d.key for d in descriptors_proteinqc.SEQ_COMPOSITION_DESCRIPTORS]
)
print(len(all_values), 'designs')
print(len(all_values.columns), 'descriptors')
all_values.head()

8450 designs
43 descriptors


,Sequence A,Sequence length,Ala %,Cys %,Asp %,Glu %,Phe %,Gly %,His %,Ile %,...,MEC reduced,MEC cystines,Helix-forming residues %,Sheet-forming residues %,Turn-forming residues %,Flexibility average,GRAVY,Instability index,Molecular weight,Sequence entropy
design_id,,,,,,,,,,,,,,,,,,,,,
ovo_avz_0001_cycle01,SEKYSQEQYERIKKALEEEGTEEAKEALEALEESLEENEKRKKELE...,76,5.263158,0.0,0.000000,56.578947,0.000000,1.315789,0.0,1.315789,...,2980,2980,81.578947,11.842105,6.578947,1.061305,-2.485526,153.828947,9399.5491,1.896239
ovo_avz_0001_cycle02,SKEYSKELYEKILNYLKEENTEEAKKALKELEELLKELEKREKERE...,76,2.631579,0.0,0.000000,39.473684,0.000000,0.000000,0.0,1.315789,...,4470,4470,80.263158,17.105263,5.263158,1.055339,-2.481579,76.990789,9673.7253,2.001427
ovo_avz_0001_cycle03,SEEESEELYKRILEFLKEEGTEEAEEAKKLLEEFVKELKKREKERE...,76,2.631579,0.0,0.000000,52.631579,2.631579,1.315789,0.0,1.315789,...,1490,1490,82.894737,15.789474,3.947368,1.057714,-2.365789,133.507895,9553.1232,1.974242
ovo_avz_0001_cycle04,SREESLKLYESILEYLDKEGTKEAEEAKKLLEELKKELEEREEKRE...,76,2.631579,0.0,1.315789,56.578947,0.000000,1.315789,0.0,1.315789,...,2980,2980,81.578947,15.789474,6.578947,1.056960,-2.357895,140.994737,9525.7916,1.745359
ovo_avz_0002_cycle01,SKEEELLEIFDELEKAEKEYKKLKEEAEKKEKELKEKAKKSNEEEK...,85,5.882353,0.0,2.352941,36.470588,1.176471,1.176471,0.0,3.529412,...,1490,1490,84.705882,21.176471,8.235294,1.051152,-1.665882,52.664706,10221.4766,2.115333


In [14]:
all_values.to_csv('../../data/results/04_rfdiffusion_binder_design/seq_descriptors_all_designs.csv.gz')